Develop the process to download (only) the English book texts using Python. Find a suitable way to store the results for further processing (which you will do in assignment 2), e.g. saving the downloaded documents to a dedicated folder.
Besides the documents, you need to store some properties of all downloaded documents. You should at least store the following properties of the downloaded documents as shown on the website:

### Property
- URL (of plaintext version)
- Title
- Author
- Reading Level
- EBook-No.
- Language
- Most Recently Updated


### To Do: 
1. eerst linken naar de goede plaats: onder titel van Top 100 EBooks last 30 days, ik denk dat je dat kan zoeken door te kijken naar de titel van html ofzo? GEDAAN
2. daarna moeten de links geopend worden naar het boek GEDAAN
3. de properties moeten worden worden opgeslagen in een document GEDAAN
4. Er moet gecheckt worden of property compleet is (dit staat op de pagina zelf, niet in het boek), anders verwijderen uit rij. (dus ook of het engels is of niet) 
5. boek moet worden geopend in de link 'Plain Text UTF-8' (als die er is)
6. de boeken moeten gedownload worden in een mapje



In [7]:
import re
from urllib.request import urlopen, urljoin
from bs4 import BeautifulSoup
import pandas as pd
import requests 
import os
import time

In [ ]:
# URL of the Top 100 EBooks (last 30 days)
url = "https://books.flotwiskunde.nl/browse/scores/top.html"

# Create folder for downloaded books
download_folder = "downloaded_books"
os.makedirs(download_folder, exist_ok=True)

# Open the webpage and read its HTML content
page = urlopen(url)
html = page.read()

# Parse the HTML using BeautifulSoup
soup = BeautifulSoup(html, "html.parser")

# Find the section header for "Top 100 EBooks last 30 days"
header = soup.find("h2", id="books-last30")

# The ordered list (<ol>) directly after this header contains the books
book_list = header.find_next("ol")


# Step 1: Collect links to all book detail pages
links = []

# Loop over all list items in the ordered list
for li in book_list.find_all("li"):
    a_tag = li.find("a")  # Find the hyperlink inside the list item
    if a_tag:
        link = a_tag.get("href")  # Extract relative link
        full_link = urljoin(url, link)  # Convert to absolute URL
        links.append(full_link)  # Store full URL


# Step 2: Visit each book page and extract metadata
book_dicts = []

for web_url in links:
    
    # Send HTTP request to book page
    response = requests.get(web_url)
    soup = BeautifulSoup(response.text, "html.parser")
        
    # Find the metadata table (class 'bibrec')
    table_2 = soup.find('table', class_='bibrec')
    
    # Temporary dictionary to store properties of one book
    book_info = {}
    
    if table_2:
        # Loop over all rows in the metadata table
        for row in table_2.find_all('tr'):
            
            header = row.find('th')  # Property name
            value = row.find('td')   # Property value
            
            if header and value:
                key = header.text.strip()
                val = value.text.strip()
                
                # Store only the required properties
                if key == "Title":
                    book_info["Title"] = val
                elif key == "Author":
                    book_info["Author"] = val
                elif key == "Reading Level":
                    book_info["Reading_Level"] = val
                elif key == "EBook-No.":
                    book_info["EBook_No."] = val
                elif key == "Language":
                    book_info["Language"] = val
                elif key == "Most Recently Updated":
                    book_info["Most_Recently_Updated"] = val
            
        # Only add English books        
        if book_info.get("Language") != "English":
            continue
            
        # Check if all required fields are present
        required_fields = ["Title", "Author", "Reading_Level", "EBook_No.", "Language", "Most_Recently_Updated"]

        if not all(book_info.get(field) for field in required_fields):
            continue

        # Look for download link for "Plain Text UTF-8"
        plain_text_link = None

        for link in soup.find_all("a"):
            if link.text and "Plain Text UTF-8" in link.text:
                plain_text_link = urljoin(web_url, link.get("href"))
                break

        # Skip if no plain text version is available
        if not plain_text_link:
            continue
        
        # Download book
        try:
            response = requests.get(plain_text_link)
    
        # filename = EBook-number.txt
            filename = os.path.join(download_folder, f"{book_info['EBook_No.']}.txt")
    
            with open(filename, "w", encoding="utf-8") as f:
                f.write(response.text)

        except Exception as e:
            print(f"Something went wrong while trying to download {web_url}: {e}")
            continue

        # Append extracted data to main list
        # .get() ensures no KeyError if a property is missing
        book_dicts.append({
            "URL": web_url,
            "Title": book_info.get("Title", ""),
            "Author": book_info.get("Author", ""),
            "Reading_Level": book_info.get("Reading_Level", ""),
            "EBook_No.": book_info.get("EBook_No.", ""),
            "Language": book_info.get("Language", ""),
            "Most_Recently_Updated": book_info.get("Most_Recently_Updated", "")
        })


# Step 3: Convert results to a pandas DataFrame
properties_books = pd.DataFrame(book_dicts)

# Save metadata as CSV
# properties_books.to_csv("book_metadata.csv", index=False)

# Bronnen
- https://www.w3schools.com/python/python_regex.asp
- https://realpython.com/python-web-scraping-practical-introduction/
- https://www.slingacademy.com/article/extract-all-links-from-a-webpage-using-python-and-beautiful-soup/?utm_source=chatgpt.com
- https://memudualimatou.medium.com/web-scraping-how-to-extract-a-table-8c979ab40c23
- https://docs.python.org/3/library/os.path.html#os.path.join
- https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files
- https://docs.python-requests.org/en/latest/user/quickstart/#response-content

# Takenverdeling
### Yvonne
- Basis webscraper opgezet.
- Links van de Top 100 pagina verzameld.
- Metadata (Title, Author, Reading Level, EBook-No., Language, Most Recently Updated) per boek gescrapet.
- Gegevens opgeslagen in een pandas DataFrame.

### Jessie
- Filtering toegevoegd voor alleen Engelstalige boeken.
- Controle ingebouwd op complete metadata.
- “Plain Text UTF-8” downloadlink automatisch laten zoeken.
- Boeken gedownload en opgeslagen in aparte map.
- Metadata opgeslagen als CSV-bestand voor verdere verwerking.

# AI gebruik
- Nagaan of de code goed functioneert
- Errors verwerken
- Gevraagd naar specifieke tutorials over bepaalde onderwerpen
- Matchen van elke specifieke kopje van de tabel
- Comments en docstrings laten toevoegen